In [1]:
import numpy as np
from scipy import stats

F = {
 "ConvNeXt-Tiny":    [0.7761, 0.7876, 0.7345, 0.7803, 0.7221],
 "ResNet50":         [0.7517, 0.7612, 0.7413, 0.7625, 0.7007],
 "EfficientNet-B0":  [0.7516, 0.7403, 0.7202, 0.7527, 0.7498],
 "Swin-Tiny":        [0.7522, 0.7304, 0.7742, 0.7534, 0.6983],
 "MobileNetV3-Large":[0.7071, 0.7498, 0.7595, 0.7381, 0.6969],
}
base, rows = "ConvNeXt-Tiny", []
for m, v in F.items():
    if m == base: continue
    d = np.array(F[base]) - np.array(v)
    t, p = stats.ttest_rel(F[base], v)
    h = stats.t.ppf(0.975, len(d)-1) * d.std(ddof=1) / np.sqrt(len(d))
    rows.append([m, d.mean(), d.mean()-h, d.mean()+h, p])

# Holm correction
order = np.argsort([r[4] for r in rows]); k = len(rows); run = 0
for rank, i in enumerate(order):
    run = max(run, min(1.0, (k - rank) * rows[i][4]))
    rows[i].append(run)

print(f"{'vs ConvNeXt':<20}{'meanΔF1':>9}{'CI low':>9}{'CI high':>9}{'p':>8}{'p_Holm':>8}")
for r in rows:
    print(f"{r[0]:<20}{r[1]:>9.4f}{r[2]:>9.4f}{r[3]:>9.4f}{r[4]:>8.3f}{r[5]:>8.3f}")

vs ConvNeXt           meanΔF1   CI low  CI high       p  p_Holm
ResNet50               0.0166  -0.0001   0.0334   0.051   0.204
EfficientNet-B0        0.0172  -0.0173   0.0517   0.239   0.478
Swin-Tiny              0.0184  -0.0255   0.0624   0.309   0.478
MobileNetV3-Large      0.0298  -0.0131   0.0728   0.126   0.377


In [ ]:
import sys, numpy as np, torch, torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import models
from sklearn.metrics import f1_score, accuracy_score
sys.path.append("../src")
from dataset import gather_samples, stratified_split, RiceLeafDataset, CLASSES

dev = "cuda" if torch.cuda.is_available() else "cpu"
N = len(CLASSES)
_, _, test_set = stratified_split(gather_samples())
loader = DataLoader(RiceLeafDataset(test_set, "test"), batch_size=32, shuffle=False, num_workers=2)
y = np.array([l for _, l in test_set])

def build(name):
    if name == "ResNet50":
        m = models.resnet50(weights=None); m.fc = nn.Linear(m.fc.in_features, N)
    elif name == "EfficientNet-B0":
        m = models.efficientnet_b0(weights=None); m.classifier[1] = nn.Linear(m.classifier[1].in_features, N)
    elif name == "MobileNetV3-Large":
        m = models.mobilenet_v3_large(weights=None); m.classifier[3] = nn.Linear(m.classifier[3].in_features, N)
    elif name == "ConvNeXt-Tiny":
        m = models.convnext_tiny(weights=None); m.classifier[2] = nn.Linear(m.classifier[2].in_features, N)
    else:
        m = models.swin_t(weights=None); m.head = nn.Linear(m.head.in_features, N)
    return m

CK = {"ResNet50": "resnet50_best.pth", "EfficientNet-B0": "efficientnet_b0_best.pth",
      "MobileNetV3-Large": "mobilenetv3_large_best.pth", "ConvNeXt-Tiny": "convnext_tiny_best.pth",
      "Swin-Tiny": "swin_tiny_best.pth"}

probs = {}
for name, ck in CK.items():
    m = build(name)
    sd = torch.load(ck, map_location="cpu")
    if isinstance(sd, dict) and "model_state_dict" in sd: sd = sd["model_state_dict"]
    m.load_state_dict(sd); m.to(dev).eval()
    out = []
    with torch.no_grad():
        for x, _ in loader:
            out.append(torch.softmax(m(x.to(dev)), 1).cpu().numpy())
    probs[name] = np.concatenate(out)
    del m; torch.cuda.empty_cache()

probs["Ensemble (5 models)"] = np.mean([probs[k] for k in CK], axis=0)

rng = np.random.default_rng(42)
idx = [rng.integers(0, len(y), len(y)) for _ in range(1000)]
print(f"{'model':<22}{'Acc':>7}{'MacroF1':>9}{'CI low':>8}{'CI high':>8}")
for k, p in probs.items():
    pred = p.argmax(1)
    f = [f1_score(y[i], pred[i], average="macro") for i in idx]
    lo, hi = np.percentile(f, [2.5, 97.5])
    print(f"{k:<22}{accuracy_score(y, pred):>7.4f}{f1_score(y, pred, average='macro'):>9.4f}{lo:>8.4f}{hi:>8.4f}")